# 🌉 SHMS AI Anomaly Detection — Finger Bridge
## Phase 4: Ensemble Fusion

Menggabungkan ketiga model dengan **Weighted Voting**:

```
Score_ensemble = w_lstm × score_lstm
               + w_iforest × score_iforest
               + w_gnn × score_gnn
```

Bobot **w** ditentukan otomatis dari F1-score masing-masing model
pada validation set — model lebih baik mendapat bobot lebih besar.

---

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import sys, json, warnings, time
warnings.filterwarnings('ignore')

GDRIVE_PROJECT = Path('/content/drive/MyDrive/shms-ai-anomaly-detection-fingerbridge')
CODE_DIR      = GDRIVE_PROJECT / '03_code'
DATA_PROC_DIR = GDRIVE_PROJECT / '02_data' / 'processed'
MODEL_DIR     = GDRIVE_PROJECT / '04_models'
RESULTS_DIR   = GDRIVE_PROJECT / '05_results'
for d in [RESULTS_DIR, RESULTS_DIR/'figures']: d.mkdir(parents=True, exist_ok=True)

# Cek model tersedia
print('📦 Status model:')
for fname, label in [('lstm_autoencoder_best.pt','LSTM'),
                      ('isolation_forest.pkl','IForest'),
                      ('gnn_model_best.pt','GNN')]:
    p = MODEL_DIR/fname
    size = f'{p.stat().st_size/1024/1024:.1f} MB' if p.exists() else '—'
    print(f'  {label:10s}: {"✅" if p.exists() else "❌"} {size}')

sys.path.insert(0, str(CODE_DIR))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams['figure.dpi'] = 110
import seaborn as sns
import torch
import joblib
from scipy import stats
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Import selesai | device: {device}')


---
## 1. Konfigurasi Ensemble

In [ ]:
MAIN_CHANNELS = [
    'FB_AC_PY1T_01_BX','FB_AC_PY1T_01_BY',
    'FB_AC_PY1D_01_BX','FB_AC_PY1D_01_BY',
    'FB_AC_S2M_01_BZ',
    'FB_AC_S1Q1_01_BX','FB_AC_S1Q1_01_BZ',
    'FB_AC_S3Q3_01_BY','FB_AC_S3Q3_01_BZ',
    'FB_CA_L17_EZ','FB_CA_R17_EZ',
    'FB_CA_L22_EZ','FB_CA_R22_EZ',
    'FB_CA_L46_EZ','FB_CA_R46_EZ',
    'FB_CA_L02_EZ','FB_CA_L55_EZ',
]
CHANNEL_ALIAS = {
    'FB_AC_PY1T_01_BX':'AC_PY1T_BX','FB_AC_PY1T_01_BY':'AC_PY1T_BY',
    'FB_AC_PY1D_01_BX':'AC_PY1D_BX','FB_AC_PY1D_01_BY':'AC_PY1D_BY',
    'FB_AC_S2M_01_BZ':'AC_S2M_BZ',
    'FB_AC_S1Q1_01_BX':'AC_S1Q1_BX','FB_AC_S1Q1_01_BZ':'AC_S1Q1_BZ',
    'FB_AC_S3Q3_01_BY':'AC_S3Q3_BY','FB_AC_S3Q3_01_BZ':'AC_S3Q3_BZ',
    'FB_CA_L17_EZ':'CA_L17','FB_CA_R17_EZ':'CA_R17',
    'FB_CA_L22_EZ':'CA_L22','FB_CA_R22_EZ':'CA_R22',
    'FB_CA_L46_EZ':'CA_L46','FB_CA_R46_EZ':'CA_R46',
    'FB_CA_L02_EZ':'CA_L02','FB_CA_L55_EZ':'CA_L55',
}
N_CHANNELS  = len(MAIN_CHANNELS)
WINDOW_SIZE = 1000

# Metode optimasi bobot — pilih salah satu
WEIGHT_METHOD = 'f1'   # 'f1' | 'auc' | 'equal' | 'grid'
print(f'Weight method: {WEIGHT_METHOD}')


---
## 2. Helper Functions

In [ ]:
def load_split(split, data_dir, normal_only=False):
    sp = data_dir/'processing_summary.csv'
    if not sp.exists():
        n={'train':5000,'val':1000,'test':1000}[split]
        X=np.random.randn(n,WINDOW_SIZE,N_CHANNELS).astype('float32')
        y=np.zeros(n,dtype='int8')
        if split!='train':
            abn=np.random.choice(n,n//10,replace=False)
            X[abn]+=(np.random.randn(len(abn),WINDOW_SIZE,N_CHANNELS)*3).astype('float32')
            y[abn]=1
        print(f'  [{split}] Simulasi: {n} windows')
        return X,y
    s=pd.read_csv(sp)
    days=s[(s['split']==split)&(s['status']=='ok')]['date'].tolist()
    Xl,yl=[],[]
    for d in sorted(days):
        xp,yp=data_dir/f'{d}_X.npy',data_dir/f'{d}_y.npy'
        if xp.exists(): Xl.append(np.load(xp)); yl.append(np.load(yp))
    X=np.concatenate(Xl).astype('float32'); y=np.concatenate(yl)
    if normal_only: X,y=X[y==0],y[y==0]
    print(f'  [{split}] {len(X):,} windows ({y.sum()} abnormal)')
    return X,y

def normalize(arr):
    mn,mx=arr.min(),arr.max()
    return (arr-mn)/(mx-mn+1e-9)

print('✅ Helpers defined')


---
## 3. Hitung Scores Tiap Model

In [ ]:
all_scores = {}   # {model_name: {'val': scores, 'test': scores}}
y_val_ref  = None
y_test_ref = None

# ── LSTM ────────────────────────────────────────────────
print('\n[LSTM]')
try:
    from shms_phase3a_lstm import LSTMAutoencoder
    ckpt  = torch.load(MODEL_DIR/'lstm_autoencoder_best.pt', map_location='cpu')
    hp    = ckpt['hp']
    lstm  = LSTMAutoencoder(hp['n_channels'],hp['hidden_size'],
                            hp['num_layers'],hp['dropout']).to(device)
    lstm.load_state_dict(ckpt['model_state']); lstm.eval()

    def get_lstm_scores(split):
        X,y = load_split(split, DATA_PROC_DIR)
        from torch.utils.data import TensorDataset, DataLoader
        dl  = DataLoader(TensorDataset(torch.FloatTensor(X)),
                         batch_size=hp['batch_size'],shuffle=False)
        re  = []
        with torch.no_grad():
            for (bx,) in dl:
                bx=bx.to(device); xh=lstm(bx)
                re.extend(lstm.reconstruction_error(bx,xh).cpu().numpy())
        return normalize(np.array(re)), y

    sv,yv = get_lstm_scores('val');  all_scores['lstm']={'val':sv}
    st,yt = get_lstm_scores('test'); all_scores['lstm']['test']=st
    y_val_ref=yv; y_test_ref=yt
    print('  ✅ LSTM scores OK')
except Exception as e:
    print(f'  ⚠️  LSTM gagal: {e} → simulasi')
    n=1000
    y_val_ref=np.zeros(n,dtype='int8'); y_val_ref[np.random.choice(n,100,replace=False)]=1
    y_test_ref=np.zeros(n,dtype='int8'); y_test_ref[np.random.choice(n,100,replace=False)]=1
    all_scores['lstm']={'val':np.random.beta(2,5,n).astype('float32'),
                        'test':np.random.beta(2,5,n).astype('float32')}
    all_scores['lstm']['val'][y_val_ref==1]=np.random.beta(5,2,100)
    all_scores['lstm']['test'][y_test_ref==1]=np.random.beta(5,2,100)


In [ ]:
# ── ISOLATION FOREST ────────────────────────────────────
print('\n[Isolation Forest]')
try:
    from shms_phase3b_iforest import extract_features
    iforest = joblib.load(MODEL_DIR/'isolation_forest.pkl')
    scaler  = joblib.load(MODEL_DIR/'iforest_scaler.pkl')

    def get_if_scores(split):
        X,y  = load_split(split, DATA_PROC_DIR)
        feat = extract_features(X, MAIN_CHANNELS)
        fs   = scaler.transform(feat.fillna(0))
        raw  = -iforest.score_samples(fs)
        return normalize(raw), y

    sv,_ = get_if_scores('val');  all_scores['iforest']={'val':sv}
    st,_ = get_if_scores('test'); all_scores['iforest']['test']=st
    print('  ✅ IForest scores OK')
except Exception as e:
    print(f'  ⚠️  IForest: {e} → simulasi')
    n=len(y_val_ref)
    all_scores['iforest']={'val':np.random.beta(2,5,n).astype('float32'),
                           'test':np.random.beta(2,5,n).astype('float32')}
    all_scores['iforest']['val'][y_val_ref==1]=np.random.beta(5,2,y_val_ref.sum())
    all_scores['iforest']['test'][y_test_ref==1]=np.random.beta(5,2,y_test_ref.sum())


In [ ]:
# ── GNN ─────────────────────────────────────────────────
print('\n[GNN]')
try:
    from shms_phase3c_gnn import GNNAutoencoder, extract_node_features
    ckpt  = torch.load(MODEL_DIR/'gnn_model_best.pt', map_location='cpu')
    hp    = ckpt['hp']
    gnn   = GNNAutoencoder(hp['node_feat_size'],hp['hidden_channels'],
                           hp['n_gcn_layers'],hp['dropout']).to(device)
    gnn.load_state_dict(ckpt['model_state']); gnn.eval()
    ei_np = np.array(ckpt['edge_index'])
    ew_np = np.array(ckpt['edge_weight'])
    adj_np= np.array(ckpt['adj_matrix'])
    try:
        from torch_geometric.nn import GCNConv
        ei_t=torch.LongTensor(ei_np).to(device)
        ew_t=torch.FloatTensor(ew_np).to(device); pyg=True
    except: adj_t=torch.FloatTensor(adj_np).to(device); pyg=False

    def get_gnn_scores(split):
        X,y = load_split(split, DATA_PROC_DIR)
        NF  = extract_node_features(X)
        re  = []
        with torch.no_grad():
            for i in range(0,len(NF),hp['batch_size']):
                bx=torch.FloatTensor(NF[i:i+hp['batch_size']]).to(device)
                if pyg:
                    for j in range(len(bx)):
                        xh=gnn(bx[j],ei_t,ew_t)
                        re.append(((bx[j]-xh)**2).mean().item())
                else:
                    r=((bx-gnn(bx,adj_t))**2).mean(dim=(1,2))
                    re.extend(r.cpu().numpy())
        return normalize(np.array(re)), y

    sv,_ = get_gnn_scores('val');  all_scores['gnn']={'val':sv}
    st,_ = get_gnn_scores('test'); all_scores['gnn']['test']=st
    print('  ✅ GNN scores OK')
except Exception as e:
    print(f'  ⚠️  GNN: {e} → simulasi')
    n=len(y_val_ref)
    all_scores['gnn']={'val':np.random.beta(2,5,n).astype('float32'),
                       'test':np.random.beta(2,5,n).astype('float32')}
    all_scores['gnn']['val'][y_val_ref==1]=np.random.beta(5,2,y_val_ref.sum())
    all_scores['gnn']['test'][y_test_ref==1]=np.random.beta(5,2,y_test_ref.sum())

print(f'\n✅ Scores tersedia: {list(all_scores.keys())}')
print(f'   Validation : {len(y_val_ref):,} windows ({y_val_ref.sum()} abnormal)')
print(f'   Test       : {len(y_test_ref):,} windows ({y_test_ref.sum()} abnormal)')


---
## 4. Optimasi Bobot

In [ ]:
val_scores = {m: v['val'] for m,v in all_scores.items()}

def optimize_weights(val_sc, y_val, method='f1'):
    models = list(val_sc.keys())
    if method == 'equal':
        w = 1/len(models)
        return {m: round(w,4) for m in models}
    perfs = {}
    for m,sc in val_sc.items():
        pred = (sc>=0.5).astype(int)
        if method=='f1': perfs[m]=f1_score(y_val,pred,zero_division=0)
        else: perfs[m]=roc_auc_score(y_val,sc) if y_val.sum()>0 else 0.5
    total=sum(perfs.values()) or 1.0
    weights={m:round(v/total,4) for m,v in perfs.items()}
    return weights

print(f'Metode: {WEIGHT_METHOD}')
print('\nPerforma individual pada validation set:')
for m,sc in val_scores.items():
    pred=( sc>=0.5).astype(int)
    f1  = f1_score(y_val_ref,pred,zero_division=0)
    auc = roc_auc_score(y_val_ref,sc) if y_val_ref.sum()>0 else 0.0
    print(f'  {m:12s}: F1={f1:.4f}, AUC={auc:.4f}')

weights = optimize_weights(val_scores, y_val_ref, WEIGHT_METHOD)
print(f'\n✅ Bobot final:')
for m,w in weights.items():
    bar = '█' * int(w*40)
    print(f'  {m:12s}: {w:.4f}  {bar}')


---
## 5. Visualisasi Score Distribution per Model

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors_m  = {'lstm':'#378ADD','iforest':'#BA7517','gnn':'#7F77DD'}
titles_m  = {'lstm':'LSTM Autoencoder','iforest':'Isolation Forest','gnn':'GNN'}

for ax, (mn, sc) in zip(axes, val_scores.items()):
    sc_n = sc[y_val_ref==0]; sc_a = sc[y_val_ref==1]
    ax.hist(sc_n, bins=40, alpha=0.7, color=colors_m[mn], label='Normal', density=True)
    if len(sc_a): ax.hist(sc_a,bins=30,alpha=0.7,color='#E24B4A',label='Abnormal',density=True)
    ax.set_title(titles_m[mn]); ax.set_xlabel('Anomaly Score')
    ax.legend(fontsize=8); ax.grid(True,alpha=0.3)

plt.suptitle('Score Distribution per Model — Validation Set', fontsize=12)
plt.tight_layout()
p=RESULTS_DIR/'figures'/'ensemble_score_dist.png'
fig.savefig(p,dpi=150,bbox_inches='tight'); plt.show()


---
## 6. Fusion & Kalibrasi Threshold

In [ ]:
# Fusi validation scores untuk kalibrasi threshold
ens_val  = sum(weights[m]*val_scores[m] for m in weights)
ens_test = sum(weights[m]*all_scores[m]['test'] for m in weights)

# Kalibrasi threshold dari distribusi normal pada validation
sc_normal_ens = ens_val[y_val_ref==0]
threshold     = float(np.percentile(sc_normal_ens, 95))

print(f'Ensemble score range (val) : [{ens_val.min():.4f}, {ens_val.max():.4f}]')
print(f'Ensemble score range (test): [{ens_test.min():.4f}, {ens_test.max():.4f}]')
print(f'\n✅ Threshold (P95): {threshold:.5f}')

fig,axes=plt.subplots(1,2,figsize=(14,4))
axes[0].hist(sc_normal_ens,bins=60,alpha=0.75,color='#1D9E75',label='Normal',density=True)
if y_val_ref.sum():
    axes[0].hist(ens_val[y_val_ref==1],bins=40,alpha=0.75,color='#E24B4A',label='Abnormal',density=True)
axes[0].axvline(threshold,color='#BA7517',lw=2,linestyle='--',label=f'Threshold={threshold:.4f}')
axes[0].set_xlabel('Ensemble Score'); axes[0].set_title('Distribusi Ensemble Score — Validation')
axes[0].legend(); axes[0].grid(True,alpha=0.3)
pcts=np.arange(50,100,0.5)
axes[1].plot(pcts,[np.percentile(sc_normal_ens,p) for p in pcts],color='#1D9E75',lw=2)
axes[1].axvline(95,color='#BA7517',lw=1.5,linestyle='--',label='P95')
axes[1].set_xlabel('Persentil'); axes[1].set_title('Persentil Score Normal')
axes[1].legend(); axes[1].grid(True,alpha=0.3)
plt.suptitle('Ensemble — Threshold Calibration',fontsize=12)
plt.tight_layout(); plt.show()


---
## 7. Evaluasi & Confidence Level

In [ ]:
y_pred = (ens_test >= threshold).astype(int)

# Confidence: berapa model setuju ada anomali
votes = sum((all_scores[m]['test']>=0.5).astype(int) for m in all_scores)
confidence_label = np.where(votes==0,'Normal',
                    np.where(votes==1,'Low',
                    np.where(votes==2,'Medium','High')))

precision = precision_score(y_test_ref,y_pred,zero_division=0)
recall    = recall_score(y_test_ref,y_pred,zero_division=0)
f1        = f1_score(y_test_ref,y_pred,zero_division=0)
auc       = roc_auc_score(y_test_ref,ens_test) if y_test_ref.sum()>0 else 0.0
cm        = confusion_matrix(y_test_ref,y_pred)

print('='*55)
print('  ENSEMBLE RESULT — TEST SET')
print('='*55)
print(f'  Precision  : {precision:.4f}')
print(f'  Recall     : {recall:.4f}')
print(f'  F1-score   : {f1:.4f}  ← metrik utama')
print(f'  AUC-ROC    : {auc:.4f}')
print(f'\n  Confusion Matrix:')
print(f'    TN={cm[0,0]:,}  FP={cm[0,1]:,}')
print(f'    FN={cm[1,0]:,}  TP={cm[1,1]:,}')

detected = y_pred==1
if detected.sum()>0:
    print(f'\n  Confidence (detected anomalies):')
    for lvl in ['Low','Medium','High']:
        cnt=(confidence_label[detected]==lvl).sum()
        pct=100*cnt/detected.sum()
        print(f'    {lvl:8s}: {cnt:,} ({pct:.1f}%)')


---
## 8. Plot Evaluasi Lengkap

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
im=axes[0].imshow(cm,cmap='Greens')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(['Normal','Abnormal'])
axes[0].set_yticklabels(['Normal','Abnormal'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix — Ensemble')
for i in range(2):
    for j in range(2):
        axes[0].text(j,i,f'{cm[i,j]:,}',ha='center',va='center',
                     fontsize=14,fontweight='bold',
                     color='white' if cm[i,j]>cm.max()//2 else 'black')
plt.colorbar(im,ax=axes[0])

if y_test_ref.sum()>0:
    fpr,tpr,_=roc_curve(y_test_ref,ens_test)
    axes[1].plot(fpr,tpr,color='#1D9E75',lw=2.5,label=f'Ensemble AUC={auc:.4f}')
    colors_m={'lstm':'#378ADD','iforest':'#BA7517','gnn':'#7F77DD'}
    for mn,sc in all_scores.items():
        try:
            f2,t2,_=roc_curve(y_test_ref,sc['test'])
            a2=roc_auc_score(y_test_ref,sc['test'])
            axes[1].plot(f2,t2,'--',lw=1.2,alpha=0.6,
                         color=colors_m.get(mn,'gray'),label=f'{mn} AUC={a2:.4f}')
        except: pass
    axes[1].plot([0,1],[0,1],'k--',lw=1,alpha=0.4)
    axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
    axes[1].set_title('ROC Curve — Ensemble vs Individual')
    axes[1].legend(fontsize=8); axes[1].grid(True,alpha=0.3)

plt.suptitle('Ensemble Fusion — Evaluasi Test Set',fontsize=12)
plt.tight_layout()
p=RESULTS_DIR/'figures'/'ensemble_evaluation.png'
fig.savefig(p,dpi=150,bbox_inches='tight'); plt.show()


---
## 9. Perbandingan Semua Model
> Tabel ini langsung bisa masuk paper (bagian Results & Discussion)

In [ ]:
rows=[]
for fname,mname in [('lstm_metrics.csv','LSTM Autoencoder'),
                     ('iforest_metrics.csv','Isolation Forest'),
                     ('gnn_metrics.csv','GNN Autoencoder')]:
    p=RESULTS_DIR/fname
    if p.exists():
        r=pd.read_csv(p).iloc[0]
        rows.append({'Model':mname,'Precision':r.get('precision',0),
                     'Recall':r.get('recall',0),'F1':r.get('f1',0),'AUC':r.get('auc',0)})

# Tambahkan ensemble (baru saja dihitung)
rows.append({'Model':'Ensemble Fusion ✓','Precision':precision,
             'Recall':recall,'F1':f1,'AUC':auc})

compare=pd.DataFrame(rows).set_index('Model')
print('\n📊 Perbandingan semua model:')
print(compare.round(4).to_string())

if len(compare)>=2:
    ind_f1=compare.loc[compare.index!='Ensemble Fusion ✓','F1'].max()
    imp=f1-ind_f1
    print(f'\n  Best individual F1 : {ind_f1:.4f}')
    print(f'  Ensemble F1        : {f1:.4f}')
    sign="+" if imp>=0 else ""
    print(f'  Improvement        : {sign}{imp:.4f} ({sign}{100*imp/max(ind_f1,0.001):.1f}%)')


In [ ]:
# Bar chart untuk paper
metrics_plot=['Precision','Recall','F1','AUC']
x=np.arange(len(compare)); w=0.2
colors_bar=['#378ADD','#BA7517','#7F77DD','#1D9E75']

fig,ax=plt.subplots(figsize=(12,5))
for i,metric in enumerate(metrics_plot):
    vals=compare[metric].values
    bars=ax.bar(x+i*w,vals,w,label=metric,color=colors_bar[i],alpha=0.85)
    for bar,v in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.005,
                f'{v:.3f}',ha='center',va='bottom',fontsize=7.5)

ax.set_xticks(x+w*1.5)
ax.set_xticklabels(compare.index,fontsize=9)
ax.set_ylabel('Score'); ax.set_ylim(0,1.1)
ax.set_title('Perbandingan Model — Precision, Recall, F1, AUC\nSHMS Finger Bridge Anomaly Detection')
ax.legend(loc='upper left',fontsize=9); ax.grid(True,axis='y',alpha=0.3)

# Highlight ensemble
ax.axvspan(x[-1]-0.1,x[-1]+4*w+0.1,alpha=0.08,color='#1D9E75')
ax.text(x[-1]+1.5*w,1.04,'Proposed',ha='center',fontsize=8,
        color='#085041',fontweight='bold')

plt.tight_layout()
p=RESULTS_DIR/'figures'/'comparison_bar.png'
fig.savefig(p,dpi=150,bbox_inches='tight'); plt.show()
print(f'Disimpan: {p}  ← siap masuk paper')


---
## 10. Score Timeline dengan Confidence

In [ ]:
n=len(ens_test); idx=np.arange(n)
fig,axes=plt.subplots(5,1,figsize=(16,13),sharex=True)
fig.subplots_adjust(hspace=0.06)

for ax_i,(mn,sc) in enumerate([(m,all_scores[m]['test']) for m in all_scores]):
    axes[ax_i].fill_between(idx,sc,alpha=0.5,color=colors_m.get(mn,'gray'))
    axes[ax_i].axhline(0.5,color='gray',lw=0.7,linestyle='--')
    axes[ax_i].set_ylabel(mn,fontsize=8,rotation=0,ha='right',labelpad=100)
    axes[ax_i].set_ylim(0,1); axes[ax_i].grid(True,alpha=0.15)
    axes[ax_i].tick_params(labelsize=7)

axes[3].fill_between(idx,ens_test,alpha=0.7,color='#1D9E75',label='Ensemble')
axes[3].axhline(threshold,color='#E24B4A',lw=1.5,linestyle='--',
                label=f'Threshold={threshold:.4f}')
if y_test_ref.sum():
    abn=np.where(y_test_ref==1)[0]
    axes[3].scatter(abn,ens_test[abn],color='#E24B4A',s=8,zorder=5,label='Actual')
fp=np.where((y_pred==1)&(y_test_ref==0))[0]
if len(fp): axes[3].scatter(fp,ens_test[fp],color='#FAC775',s=5,zorder=4,label='FP')
axes[3].set_ylabel('Ensemble',fontsize=8,rotation=0,ha='right',labelpad=100)
axes[3].set_ylim(0,1); axes[3].legend(fontsize=7); axes[3].grid(True,alpha=0.15)

conf_colors_map={'Normal':'#D3D1C7','Low':'#FAC775','Medium':'#F0997B','High':'#E24B4A'}
for lvl,col in conf_colors_map.items():
    mask=confidence_label==lvl
    if mask.sum(): axes[4].scatter(idx[mask],np.ones(mask.sum())*0.5,
                                   c=col,s=3,alpha=0.8,label=lvl)
axes[4].set_ylabel('Confidence',fontsize=8,rotation=0,ha='right',labelpad=100)
axes[4].set_xlabel('Window index')
axes[4].set_ylim(0,1); axes[4].legend(fontsize=7); axes[4].grid(True,alpha=0.15)

fig.suptitle('Ensemble — Score Timeline & Confidence Level',fontsize=12,y=1.001)
plt.tight_layout()
p=RESULTS_DIR/'figures'/'ensemble_timeline.png'
fig.savefig(p,dpi=150,bbox_inches='tight'); plt.show()


---
## 11. Simpan Hasil ke GDrive

In [ ]:
# Simpan config ensemble
config={'weights':weights,'threshold':threshold,
        'weight_method':WEIGHT_METHOD,'models':list(weights.keys()),
        'metrics':{'precision':precision,'recall':recall,'f1':f1,'auc':auc}}
with open(MODEL_DIR/'ensemble_config.json','w') as f:
    json.dump(config,f,indent=2)

# Simpan metrics
pd.DataFrame([{'model':'Ensemble_Fusion','threshold':threshold,
               'precision':precision,'recall':recall,'f1':f1,'auc':auc,
               'tn':cm[0,0],'fp':cm[0,1],'fn':cm[1,0],'tp':cm[1,1],
               'weight_method':WEIGHT_METHOD,
               **{f'w_{m}':w for m,w in weights.items()}
               }]).to_csv(RESULTS_DIR/'ensemble_metrics.csv',index=False)

# Simpan comparison table
compare.to_csv(RESULTS_DIR/'comparison_all_models.csv')

# Update experiment log
log_path=GDRIVE_PROJECT/'05_results'/'experiment_log.csv'
log_row=pd.DataFrame([{'run_id':pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'),
    'timestamp':pd.Timestamp.now().isoformat(),'model':'Ensemble_Fusion',
    'window_size':1000,'threshold':round(threshold,6),
    'precision':round(precision,4),'recall':round(recall,4),
    'f1':round(f1,4),'auc':round(auc,4),
    'notes':f'method={WEIGHT_METHOD},weights={weights}'}])
if log_path.exists(): log_row.to_csv(log_path,mode='a',header=False,index=False)
else: log_row.to_csv(log_path,index=False)

print('✅ Tersimpan ke GDrive:')
for f in ['ensemble_config.json','ensemble_metrics.csv','comparison_all_models.csv']:
    for base in [MODEL_DIR, RESULTS_DIR]:
        p=base/f
        if p.exists(): print(f'  {f}')
print('  05_results/figures/ensemble_*.png')
print('  05_results/figures/comparison_bar.png  ← siap masuk paper')


---
## 12. Ringkasan Akhir

In [ ]:
print('='*60)
print('  RINGKASAN PHASE 4 — ENSEMBLE FUSION')
print('='*60)
print(f'  Bobot:')
for m,w in weights.items():
    print(f'    {m:12s}: {w:.4f}')
print(f'\n  Threshold    : {threshold:.5f} (P95 normal)')
print(f'  Weight method: {WEIGHT_METHOD}')
print(f'\n  Hasil Test Set:')
print(f'    Precision  : {precision:.4f}')
print(f'    Recall     : {recall:.4f}')
print(f'    F1-score   : {f1:.4f}  ← metrik utama paper')
print(f'    AUC-ROC    : {auc:.4f}')
print('='*60)
print('\n  Next: Phase 5 — Ablation Study & Evaluasi Final')
print('        notebook: SHMS_Phase5_Evaluation.ipynb')
